In [1]:
import torch

# Check for NVIDIA GPU (CUDA)
print("CUDA Available:", torch.cuda.is_available())

# Check for Apple Silicon Mac GPU (MPS)
print("MPS Available:", torch.backends.mps.is_available())

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

CUDA Available: True
MPS Available: False
Using device: cuda


# Read the Target Sequence / Profile

In [2]:
import pandas as pd

family_df = pd.read_csv('protein_families.csv')

i = 0
name, accession, match_len, seed_len, type_, clan = family_df.iloc[i]

# Parse the sequences 

In [3]:
from Bio import SeqIO
from modules.sequences import split_train_val_test

# read the fasta files
recs = [(r.id, str(r.seq)) for r in SeqIO.parse(f"sequences/{accession}.fasta", "fasta")]

# separate to train, validation, and test set
train_seqs, val_seqs, test_seqs = split_train_val_test(recs)

# Initialize HMM

In [4]:
from modules.model import (
    init_transition_mat,
    init_emission_mat
)

# init transition and emmission matrices
trans_matdf = init_transition_mat(match_len)
emit_matdf = init_emission_mat(match_len)

# encode the symbols into index for the hmmlearn
symbols = list(emit_matdf.index)            # ^ A C D ... Y $  (taken from the CSV row order)
sym_idx = {s: i for i, s in enumerate(symbols)}
BEGIN = sym_idx["^"]                         # dummy begin symbol, emitted only by M0
END = sym_idx["$"]    

# Convert to HQMM

In [5]:
from modules.qmodel import (
    build_hqmm,
    convert_to_torch
)

K = build_hqmm(trans_matdf, emit_matdf)
K_torch = convert_to_torch(K)

# Train HQMM

In [6]:
import torch
from modules.qtrain import train_hqmm
from modules.qevaluation import description_accuracy_hqmm

p = torch.zeros(K[0,1].shape, dtype=torch.complex64)
p[0, 0] = 1.0
W = 79

da0 = description_accuracy_hqmm(
    p, K_torch, W, sym_idx, test_seqs
)

results = {
    "untrained": (da0.mean(), da0.std())
}
histories = {}
trained_models = {}

settings = {
    "large step (0.1)": 0.1,
    "medium step (0.01)": 0.01,
    "small step (0.001)": 0.001,
}

max_iter = 15
tol = 1e-3
batch_size = 16
alpha = 0.8

for label, tau in settings.items():
    print(f"--- {label}: tau={tau}", flush=True)

    K_initial = {
        key: value.detach().clone()
        for key, value in K_torch.items()
    }

    K_trained, hist = train_hqmm(
        p=p,
        K=K_initial,
        W=W,
        sym_idx=sym_idx,
        train_seqs=train_seqs,
        val_seqs=val_seqs,
        test_seqs=test_seqs,
        max_iter=max_iter,
        tol=tol,
        tau=tau,
        alpha=alpha,
        batch_size=batch_size,
        verbose=True,
        device=device
    )

    # Save training results immediately
    histories[label] = hist
    trained_models[label] = K_trained

    # Align devices before evaluation
    K_ref = next(iter(K_trained.values()))

    p_eval = p.to(
        device=K_ref.device,
        dtype=K_ref.dtype
    )

    K_eval = {
        key: value.to(
            device=K_ref.device,
            dtype=K_ref.dtype
        )
        for key, value in K_trained.items()
    }

    with torch.no_grad():
        da = description_accuracy_hqmm(
            p_eval, K_eval, W, sym_idx, test_seqs
        )

    results[label] = (
        da.mean().item(),
        da.std(correction=0).item()
    )

    print(
        f"Test DA: {results[label][0]:.4f} "
        f"± {results[label][1]:.4f}",
        flush=True
    )

--- large step (0.1): tau=0.1
Training on cuda (dtype=torch.complex64)
iter   0  train     0.0s  log-lik -14656.84  val_da_mean 0.0015  test_da_mean 0.0020
  iter   1  batch   1/5  loss 227.5196  tau 0.100000
  iter   1  batch   2/5  loss 186.3331  tau 0.100000
  iter   1  batch   3/5  loss 182.4431  tau 0.100000
  iter   1  batch   4/5  loss 215.0772  tau 0.100000
  iter   1  batch   5/5  loss 140.8819  tau 0.100000
iter   1  train   132.7s  log-lik -11884.48  val_da_mean 0.1939  test_da_mean 0.2102
  iter   2  batch   1/5  loss 180.4254  tau 0.080000
  iter   2  batch   2/5  loss 154.9007  tau 0.080000
  iter   2  batch   3/5  loss 144.5572  tau 0.080000
  iter   2  batch   4/5  loss 156.3979  tau 0.080000
  iter   2  batch   5/5  loss 207.2913  tau 0.080000
iter   2  train   288.8s  log-lik -10648.76  val_da_mean 0.2541  test_da_mean 0.2705
  iter   3  batch   1/5  loss 145.1554  tau 0.064000
  iter   3  batch   2/5  loss 153.1143  tau 0.064000
  iter   3  batch   3/5  loss 150.4172

# Visualization

In [7]:
import importlib
import modules.qplots as qplots

importlib.reload(qplots)
plot_da_bars = qplots.plot_da_bars

In [13]:
import pandas as pd
import matplotlib.pyplot as plt

from modules.qplots import (
    plot_da_learning_curve,
    plot_da_bars,
)

from pathlib import Path

try:
    plt.style.use("seaborn-v0_8-darkgrid")
except OSError:
    pass

# Create directory for saved models
results_dir = Path("results/phqmm")

# Save training histories
for label, h in histories.items():
    h.to_csv(
        f"q_history_{label.split()[0]}.csv",
        index=False
    )

# Save combined training histories
combined_histories = pd.concat(
    [
        h.assign(step_size=label)
        for label, h in histories.items()
    ],
    ignore_index=True,
)

combined_histories.to_csv(
    results_dir / f"{name}_q_training_histories.csv",
    index=False,
)

# Save final description accuracy results
pd.DataFrame(
    results,
    index=["test_da_mean", "test_da_std"]
).T.to_csv(results_dir / f"{name}_q_da_results.csv")

# Optional reference DA values
references = {}

# Test DA vs GPU training time
plot_da_learning_curve(
    histories,
    references,
    x="train_time",
    out=results_dir / f"q_da_vs_time.png"
)

# Test DA vs epoch
plot_da_learning_curve(
    histories,
    references,
    x="iter",
    out=results_dir / f"q_da_vs_epoch.png"
)

# Final DA comparison by step size
plot_da_bars(
    results,
    references,
    out=results_dir / f"q_da_results.png"
)


In [12]:
histories

{'large step (0.1)':     iter   train_time        loglik  val_da_mean  val_da_std  test_da_mean  \
 0      0     0.000000 -14656.837891     0.001518    0.002323      0.001961   
 1      1   132.723745 -11884.479492     0.193858    0.090416      0.210162   
 2      2   288.808254 -10648.763672     0.254084    0.089401      0.270489   
 3      3   457.060206 -10017.844727     0.289611    0.108727      0.301290   
 4      4   604.030117  -9709.170898     0.295170    0.110707      0.308613   
 5      5   742.161194  -9448.343750     0.304878    0.120069      0.316683   
 6      6   877.534588  -9276.515625     0.308691    0.121536      0.322668   
 7      7  1010.592971  -9179.975586     0.309099    0.122385      0.324235   
 8      8  1154.567216  -9105.777344     0.311046    0.123854      0.325127   
 9      9  1297.183320  -9052.063477     0.311220    0.123413      0.325126   
 10    10  1430.548750  -9014.166992     0.310460    0.124123      0.324604   
 11    11  1566.138688  -8985.45